# DISTINCTNESS

Uses `library` (protocol and API in `library/README.md`, differences from the original implementation in `library/HISTORY.md`) and the original polluters (`library.pollution`).
Every experiment runs `run_experiment(...)` (results JSON in `Plot/results/<exp_id>.json`), then `plot_experiment(exp_id)` (figures in `Plot/<exp_id>/`), then shows the pivot of the main metric, test nRMSE (mean over seeds; lower is better).
Set `QUICK = True` (or the environment variable `DIQ_QUICK=1`) for a smoke run: x axes truncated to 2 points and one seed.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from library import pollution as pol
from library import evaluation as ev
from library.data import make_regression_data, pollute_duplicates
from library.plots import plot_experiment
from library.runner import load_results, run_experiment, summarise

SEEDS = [0, 1, 2]
N_JOBS = 8
QUICK = False  # True: x axes truncated to 2 points and SEEDS to [0] (verification only)
QUICK = QUICK or os.environ.get("DIQ_QUICK") == "1"
if QUICK:
    SEEDS = [0]

MODELS = ev.STUDY_MODELS
N_SAMPLES = 1000


def axis(values):
    values = list(values)
    return values[:2] if QUICK else values


def steps(start, stop, step):
    # inclusive range on a grid of `step`, free of float drift
    return [round(k * step, 10) for k in range(int(round(start / step)), int(round(stop / step)) + 1)]


# data generators (top level so that joblib/loky can pickle them)
def make_data_3(seed):
    return make_regression_data(N_SAMPLES, 3, 3, seed)


def make_data_9(seed):
    return make_regression_data(N_SAMPLES, 9, 4, seed)


def make_data_n(seed, x):  # x = number of rows
    return make_regression_data(int(x), 3, 3, seed)


# polluters: (X, y, x, seed) -> (X, y, meta); the polluters return copies
def pollute_distinct(X, y, x, seed):  # D1, D2: one random column
    return pol.pollution_first_second_third_experiments(X, x, seed), y, {"requested_not_distinct": x}


def pollute_distinct_fixed(X, y, x, seed):  # D3: fixed 0.2, x is the number of rows
    return pol.pollution_first_second_third_experiments(X, 0.2, seed), y, {"requested_not_distinct": 0.2, "n_rows": int(x)}


def pollute_distinct_features(X, y, x, seed):  # D4
    return pol.pollution_fourth_experiment(X, x, seed), y, {"requested_not_distinct": x}


def pollute_high_low(X, y, x, seed):  # D8
    return pol.pollution_eighth_experiment(X, x, seed), y, {"requested_not_distinct": x, "requested_not_distinct_low": 1 - x}


def pollute_informative(X, y, x, seed):  # D9
    return pol.pollution_ninth_tenth_experiments(X, y, x, informative=True, seed=seed), y, {"requested_not_distinct": x}


def pollute_less_informative(X, y, x, seed):  # D10
    return pol.pollution_ninth_tenth_experiments(X, y, x, informative=False, seed=seed), y, {"requested_not_distinct": x}


def pollute_noise(X, y, x, seed):  # D5
    return pol.pollution_fifth_experiment(X, x, seed), y, {"requested_noise_variance": x}


def pollute_bins(X, y, x, seed):  # D6
    return pol.pollution_sixth_experiment(X, int(x), seed), y, {"requested_bins": int(x)}


def pollute_outliers(X, y, x, seed):  # D7
    return pol.pollution_seventh_experiment(X, x, seed), y, {"requested_outlier_rate": x}


def show(exp_id, distinct_fraction=False):
    plot_experiment(exp_id)
    payload = load_results(exp_id)
    summ = summarise(payload)
    display(summ.pivot_table(index="x", columns=["condition", "model"], values="mean").round(4))
    if distinct_fraction:
        rec = pd.DataFrame(payload["records"])
        rec = rec[rec["condition"] == "polluted"]
        frac = pd.DataFrame(rec["meta"].map(lambda m: m["realised"]["distinct_fraction"]).tolist(), index=rec["x"])
        frac.columns = [f"feature {c}" for c in frac.columns]
        print("Mean realised distinct fraction per feature (polluted condition, all features incl. unpolluted ones, which stay at 1.0; 1.0 = all values distinct)")
        display(frac.groupby(level=0).mean().round(3))

# Section A. Distinctness of feature values

Mechanism (original polluter, report algorithm): in the chosen column a random existing value is pasted into a random other cell until `n_unique / n <= 1 - percentage`. The result is ties without missing values, so no cleaning is possible. Conditions: `polluted` (the polluted features, imputer None) and `clean` (the unpolluted reference on the same seeds, data and folds). The target is not modified. The realised distinct fraction per feature is read from the records meta and shown below each summary. At percentage 1.0 the column is clipped to one distinct value (with a warning).

## D1. Mid-high distinctness (0 to 50% not distinct)

Measures how test nRMSE changes when one random feature (3 features, all informative, n = 1000) gets 0 to 50% non-distinct values. Mechanism and conditions as in Section A.

In [ ]:
run_experiment("d1_mid_high", "Distinctness - Mid-high distinctness", "Percentage of not distinct values", axis(steps(0.0, 0.5, 0.05)),
               make_data_3, pollute_distinct, models=MODELS,
               conditions=["polluted", "clean"], seeds=SEEDS, n_jobs=N_JOBS)
show("d1_mid_high", distinct_fraction=True)

## D2. Mid-low distinctness (50 to 100% not distinct)

Measures the same effect for 50 to 100% non-distinct values of one random feature (3 features, n = 1000). At 100% the column is constant.

In [ ]:
run_experiment("d2_mid_low", "Distinctness - Mid-low distinctness", "Percentage of not distinct values", axis(steps(0.5, 1.0, 0.05)),
               make_data_3, pollute_distinct, models=MODELS,
               conditions=["polluted", "clean"], seeds=SEEDS, n_jobs=N_JOBS)
show("d2_mid_low", distinct_fraction=True)

## D3. Fixed distinctness, increasing dataset size

Measures a fixed 20% non-distinct values in one random feature (3 features) while the number of samples grows from 100 to 1000 in steps of 100. The data are generated per x with `make_data(seed, x)`, so the `clean` reference is fitted per x and the plot is a learning curve under the defect.

In [ ]:
run_experiment("d3_fixed_n", "Distinctness - Fixed distinctness of different datasets", "Number of samples", axis(steps(100, 1000, 100)),
               make_data_n, pollute_distinct_fixed, models=MODELS,
               conditions=["polluted", "clean"], seeds=SEEDS, n_jobs=N_JOBS)
show("d3_fixed_n", distinct_fraction=True)

## D4. Different percentages among features

Measures the effect when the not-distinct percentage differs between features (9 features, 4 informative, n = 1000): the original `pollution_fourth_experiment` pollutes a random subset of floor(n_features/2)+1 columns, each with a percentage drawn from N(x, 0.1) clipped to [0, 1]. x is the mean percentage, 5 to 50%.

In [ ]:
run_experiment("d4_features", "Distinctness - Different percentages among features", "Percentage of not distinct values", axis(steps(0.05, 0.5, 0.05)),
               make_data_9, pollute_distinct_features, models=MODELS,
               conditions=["polluted", "clean"], seeds=SEEDS, n_jobs=N_JOBS)
show("d4_features", distinct_fraction=True)

## D8. High and low distinctness combined

Measures the combined effect of a high-distinctness and a low-distinctness defect (3 features, n = 1000) with `pollution_eighth_experiment`, 0 to 50%: two random columns are polluted, one with x not distinct and one with 1 - x (so at x = 0.1 the second column is 90% non-distinct); the third column is untouched, and x = 0 is the clean case (original quirk). The realised-fraction table lists all columns, the unpolluted ones stay at 1.0.

In [ ]:
run_experiment("d8_high_low", "Distinctness - High and low distinctness combined", "Percentage of not distinct values", axis(steps(0.0, 0.5, 0.05)),
               make_data_3, pollute_high_low, models=MODELS,
               conditions=["polluted", "clean"], seeds=SEEDS, n_jobs=N_JOBS)
show("d8_high_low", distinct_fraction=True)

## D9. Distinctness over the most informative features

Measures the effect when the 3 most important features (random forest importance on a 80/20 split of the data, as in the original) get 0 to 50% non-distinct values (9 features, 4 informative, n = 1000).

In [ ]:
run_experiment("d9_informative", "Distinctness - Most informative features", "Percentage of not distinct values", axis(steps(0.0, 0.5, 0.05)),
               make_data_9, pollute_informative, models=MODELS,
               conditions=["polluted", "clean"], seeds=SEEDS, n_jobs=N_JOBS)
show("d9_informative", distinct_fraction=True)

## D10. Distinctness over the less informative features

Measures the same for the 3 least important features (9 features, 4 informative, n = 1000), 0 to 50%.

In [ ]:
run_experiment("d10_less_informative", "Distinctness - Less informative features", "Percentage of not distinct values", axis(steps(0.0, 0.5, 0.05)),
               make_data_9, pollute_less_informative, models=MODELS,
               conditions=["polluted", "clean"], seeds=SEEDS, n_jobs=N_JOBS)
show("d10_less_informative", distinct_fraction=True)

# Section B. Duplicate records

Record-level distinctness: exact copies of randomly chosen rows are appended so that the copies make up the requested rate of the final table (`library.data.pollute_duplicates`). Every row carries a source id (`groups`).
- `duplicated`: group-aware splitting (`GroupShuffleSplit` on source ids). A source and its copies are never split between train and test, and the test fold scores exactly one row (the original) per held-out source. Duplicates therefore only reweight the training data.
- `deduplicated`: the same splits and the same test rows, but exact duplicates are dropped from the training fold only. The comparison with `duplicated` isolates the effect of duplicates in training.
- `naive_split`: a leakage demonstration, not a recommended protocol. It uses a plain `ShuffleSplit` on all rows and scores all test rows, so copies of training rows land in the test fold and the error is optimistic.

## D11. Duplicate records (rate 0 to 50%)

Measures test nRMSE when a fraction of the table (0 to 50%, step 5%) consists of exact duplicate rows (3 features, n = 1000 original rows). Conditions: `duplicated`, `deduplicated`, `naive_split` as described above.

In [ ]:
run_experiment("d11_duplicates", "Duplicate records", "Duplicate rate", axis(steps(0.0, 0.5, 0.05)),
               make_data_3, pollute_duplicates, models=MODELS,
               conditions=["duplicated", "deduplicated", "naive_split"], seeds=SEEDS, n_jobs=N_JOBS)
show("d11_duplicates")

# Section C. Accuracy/validity (formerly distinctness 5-7)

In the original notebooks experiments 5 to 7 were filed under distinctness, but they do not create non-distinct values. Random noise (D5) is an accuracy issue, the reduction to a few bins (D6) is a loss of precision/validity of a numeric feature and outliers (D7) are invalid or extreme values. They are kept for comparison with the earlier results and relabelled; conditions are `polluted` and `clean` as in Section A.

## D5. Random noise on features

Measures the effect of adding Gaussian noise with the given variance to every feature cell (3 features, n = 1000). x is the noise variance, 0.00625 to 0.0625; the target is untouched.

In [ ]:
run_experiment("d5_noise", "Accuracy - Random noise", "Noise variance", axis([round(n * 0.00625, 10) for n in range(1, 11)]),
               make_data_3, pollute_noise, models=MODELS,
               conditions=["polluted", "clean"], seeds=SEEDS, n_jobs=N_JOBS)
show("d5_noise")

## D6. Binning of one feature

Measures the effect of reducing one random feature to a given number of bins (3 features, n = 1000), 1 to 10 bins. With 1 bin the feature is constant.

In [ ]:
run_experiment("d6_bins", "Validity - Binned feature", "Number of bins of one random feature", axis(range(1, 11)),
               make_data_3, pollute_bins, models=MODELS,
               conditions=["polluted", "clean"], seeds=SEEDS, n_jobs=N_JOBS)
show("d6_bins")

## D7. Outliers

Measures the effect of replacing one random cell in a given percentage of rows with a random integer in [0, 100) (3 features, n = 1000), 1.25 to 12.5%.

In [ ]:
run_experiment("d7_outliers", "Validity - Outliers", "Percentage of outliers", axis([round(n * 0.0125, 10) for n in range(1, 11)]),
               make_data_3, pollute_outliers, models=MODELS,
               conditions=["polluted", "clean"], seeds=SEEDS, n_jobs=N_JOBS)
show("d7_outliers")

## Differences from the original study

Differences from the original study, per experiment, and why. Common to all: the evaluation is the `library` per-fold protocol (5 shuffled folds, imputer/scalers fitted on the training fold, metrics in target units as nRMSE), data come from `make_regression_data` with 20% noise (the original used noise-free `make_regression`), there are 3 repetition seeds (0, 1, 2) in place of the single seed 2023, each experiment has an unpolluted `clean` reference on the same data and folds, the six study models are `ev.STUDY_MODELS`, and the blanket warnings filter is removed so convergence/bound flags appear in the record `status`. The original plotted one train/test curve per model; here all figures are built from the results JSON. **Band semantics:** the shaded band is mean +- SD over the 3 seeds (folds averaged within each seed first). It is descriptive, not a confidence interval.

- D1 (original exp. 1): same polluter and x axis (0 to 50%), plus `clean` reference and realised distinct fraction. Why: separates the defect effect from noise in the baseline.
- D2 (original exp. 2): same polluter and axis (50 to 100%), plus reference and realised fraction. 100% gives a constant column (clipped, with a warning).
- D3 (original exp. 3): fixed 20%, the sample size axis changed from 1000-2000 to 100-1000 step 100, and data are generated per x with `make_data(seed, x)`. Why: the original range spanned only 1000-1090 rows, too narrow for a learning curve; the wider range gives one, and the clean reference is fitted per x.
- D4 (original exp. 4): 9 features/4 informative as in the original, x axis 5 to 50% (mean percentage), with reference and realised fractions.
- D8 (original exp. 8): as in the original, with reference and realised fractions.
- D9, D10 (original exp. 9, 10): the polluters are unchanged and pick the features with a random forest importance, now seeded from the repetition seed, so the selected features can vary between seeds. Reference and realised fractions added.
- D11 (new): record-level distinctness (duplicate rows) with group-aware splitting, a deduplicated training variant and a `naive_split` leakage demonstration. Why: the original only produced ties in feature values, which is not the distinctness of records.
- D5, D6, D7 (original exp. 5, 6, 7): same polluters and axes, moved to Section C and relabelled accuracy/validity, because they do not change the distinctness of values. Reference condition added.
- Seeding: the original polluters were called with their default seed 2023; here the per-repetition pollute seed is passed, so pollution differs between seeds but is shared by all conditions and models within a seed.